# 01 – Data collection & processing
Runtime: CPU. Output goes to Drive (`DATA_DIR`).

In [5]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [6]:
import os
DATA_DIR = '/content/drive/MyDrive/s1156052/PLM-LLM/data'
OUT_DIR  = '/content/drive/MyDrive/s1156052/PLM-LLM/results'
os.makedirs(OUT_DIR, exist_ok=True)

## Sync (re-run after every git push)

In [ ]:
import os, getpass

def get_github_token():
    # 1) .env on Drive, next to DATA_DIR's parent  (GITHUB_TOKEN=...)
    env = os.path.join(os.path.dirname(DATA_DIR), '.env')
    if os.path.exists(env):
        for line in open(env):
            if line.strip().startswith('GITHUB_TOKEN='):
                return line.split('=', 1)[1].strip().strip('"\'')
    # 2) Colab Secrets (works in the browser, not in the VS Code extension)
    try:
        from google.colab import userdata
        return userdata.get('GITHUB_TOKEN')
    except Exception:
        pass
    # 3) type it once; never stored in the notebook
    return getpass.getpass('GitHub token: ')

TOKEN = get_github_token()
REPO = f'https://{TOKEN}@github.com/Tienanh209/PLM-LLM.git'
DIR = '/content/YuanZe/s1156052/PLM-LLM'
if os.path.exists(DIR):
    !git -C $DIR pull --ff-only
else:
    !git clone -q $REPO $DIR
%cd $DIR
!pip install -q -r requirements.txt

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Collect Swiss-Prot

In [ ]:
RAW = f'{DATA_DIR}/raw'; INTERIM = f'{DATA_DIR}/interim'
!python src/prepare_data.py download --out $RAW/swissprot.tsv.gz

## Cluster at 30% identity (optional, for leak-free split)

In [ ]:
# Leak-free split: cluster at 30% identity with MMseqs2
!wget -q https://mmseqs.com/latest/mmseqs-linux-avx2.tar.gz -O /tmp/mmseqs.tgz && tar xzf /tmp/mmseqs.tgz -C /tmp
import os; os.environ['PATH'] += ':/tmp/mmseqs/bin'
!python src/prepare_data.py cluster --tsv $RAW/swissprot.tsv.gz --out $INTERIM

## Process
TODO: build step (annotations -> QA jsonl + split).

In [ ]:
# TODO